# Recomendador Híbrido de Películas TMDB (2010–2025):  
## Guía de visualización · Sumativa de la Fase 4

- **Asignatura:** MCDI500 Programación para la Ciencia de Datos
- **Profesor:** Omar Salinas Silva
- **Evaluación:** Sumativa 3, Fase 4
- **Integrantes del Grupo 4:**
  1. Daniela Baeza Rocha
  2. Juan González Elgueta
  3. Elías Manríquez Armijo
  4. Yestxander Rada Gutiérrez

---

Este cuaderno es un **esqueleto** para el cuaderno de la Fase 4 de su proyecto. No es una
solución: las celdas marcadas con `TODO` las completan ustedes con su propio conjunto.
El ejemplo resuelto completo está en `proyecto-grupoX-mcdi500/F4/S3_F4_Visualizacion_Comunicacion.ipynb`.

**Cómo usarlo**

1. Cópienlo en la carpeta `F4/` de su repositorio y renómbrenlo con el formato del curso.
2. Copien también `src/comunicacion.py` del proyecto modelo a su carpeta `src/`.
3. Ejecuten antes los cuadernos de F1, F2 y F3: este cuaderno parte de los artefactos de F3.
4. Completen cada `TODO` en orden y, al final, ejecuten **Restart Kernel and Run All**.

| Sección | Apartado del informe que respalda |
|---|---|
| 1. Entorno | Reproducibilidad técnica |
| 2. Artefactos y contrato | Integración de las fases y trazabilidad |
| 3. Copia interpretable | Preparación de los datos para visualizar |
| 4. Figuras por objetivo | Resultados y visualización |
| 5. Confusión | Discusión y limitaciones |
| 6. Validación | Validación técnica |
| 7. Persistencia | Evidencias del informe y del video |

## 1. Entorno y ubicación del proyecto

In [9]:
import os
import sys
from importlib.metadata import version
import json
from datetime import date
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

SEMILLA = 42
np.random.seed(SEMILLA)


def localizar_raiz(inicio=None):
    """Raíz del proyecto aunque Jupyter abra el kernel en la carpeta F4."""
    actual = Path(inicio or Path.cwd()).resolve()
    return actual.parent if actual.name in {"F1", "F2", "F3", "F4"} else actual


RAIZ = localizar_raiz()
DIR_PROCESADO = RAIZ / "data" / "processed"
DIR_FIGURAS = RAIZ / "F4" / "figuras"
DIR_DOCS = RAIZ / "docs"
RESULTADOS = RAIZ / "F3" / "results"
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print(f"Python {sys.version.split()[0]} | entorno virtual: {sys.prefix != sys.base_prefix}")
for paquete in ("numpy", "pandas", "scipy", "scikit-learn", "matplotlib"):
    print(f"{paquete} {version(paquete)}")

Python 3.13.0 | entorno virtual: True
numpy 2.5.4
pandas 3.1.1
scipy 1.18.1
scikit-learn 1.9.1
matplotlib 3.11.2


### Configuración del proyecto

Editen **solo esta celda** para adaptar el cuaderno a su conjunto.

In [ ]:
# TODO: la variable que explican y una descripción breve de la fuente
OBJETIVO = "stroke"
FUENTE = "Fuente: Stroke Prediction Dataset (fedesoriano, Kaggle)."

# Paleta institucional: el granate señala, el resto da contexto
GRANATE, AZUL, GRIS, HIELO = "#AC212E", "#06304A", "#8A94A0", "#D7E3EE"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlelocation": "left", "axes.titleweight": "bold"})

## 2. Los artefactos de la Fase 3 y su contrato

La Fase 4 no vuelve a limpiar ni a transformar. Si falta un artefacto, el cuaderno se
detiene: no generen datos de reemplazo.

In [10]:
from src import comunicacion as com

artefactos = com.cargar_artefactos(DIR_PROCESADO)
entrenamiento = artefactos["entrenamiento"]
prueba = artefactos["prueba"]
parametros = artefactos["parametros"]

original = pd.read_csv(RAIZ / "data/raw/movies_raw.csv")
evaluacion = pd.read_csv(RESULTADOS / "evaluacion_recomendador.csv", index_col=0)
mediciones = pd.read_csv(RESULTADOS / "mediciones.csv")
for nombre in ("entrenamiento", "prueba", "diccionario"):
    
    print(f"{nombre:<14} {artefactos[nombre].shape}")

print("Parámetros cargados:", ", ".join(parametros))

entrenamiento  (14382, 16)
prueba         (3596, 16)
diccionario    (16, 3)
Parámetros cargados: semilla, fraccion_prueba, pasos, recomendador


**Pista.** Verifiquen lo que la Fase 3 prometió: mismo esquema en entrenamiento y prueba,
sin nulos y con las continuas escaladas en el entrenamiento.

In [ ]:
# TODO: al menos tres comprobaciones del contrato con assert
assert list(prueba.columns) == list(entrenamiento.columns), "Esquemas distintos"
# assert ...
# assert ...
print("Contrato verificado")

## 3. La copia interpretable

Revierte el escalamiento con los parámetros de la Fase 3 y vuelve a juntar las columnas
0/1 de cada categórica. Es solo para comunicar: los artefactos no se modifican.

In [ ]:
legible = pd.concat([com.copia_interpretable(entrenamiento, parametros),
                     com.copia_interpretable(prueba, parametros)], ignore_index=True)
print(legible.shape)
legible.head()

**Pista.** La verificación que más rinde no usa el mismo código: comparen la copia con el
archivo original de `data/raw`. Por ejemplo, ordenen una columna continua en ambos y
comprueben con `np.allclose` que coinciden.

In [ ]:
# TODO: verificación independiente contra el archivo original
# original = pd.read_csv(RAIZ / "data" / "raw" / "su_archivo.csv")
# assert np.allclose(np.sort(legible["su_continua"]), np.sort(original["su_continua"]))

## 4. Una figura por objetivo

Completen la tabla de vinculación **antes** de graficar. Cada figura responde un objetivo,
y su título dice el hallazgo.

- **Objetivo General:** Construir y evaluar un recomendador híbrido de películas, basado en similitud de contenido y calidad ponderada, sobre 17.978 películas de The Movie Database (TMDB) estrenadas entre 2010 y 2025, con un flujo reproducible y verificado.

| Objetivo específico | Objetivo específico Completo | Pregunta que responde la figura | Evidencia |
|---|---|---|---
| 1. Caracterizar el conjunto | Caracterizar la cobertura, los tipos de variables, la calidad y la distribución temporal del conjunto, e identificar qué variables sirven al recomendador | ¿Cómo se distribuyen los estrenos del catálogo? | Figura 1 ·  |
| 2. Preparar las entradas | Preparar las entradas del recomendador: tratar faltantes y ceros, y transformar los metadatos multivaluados en una sopa de metadatos vectorizada |¿Qué proporción quedó ausente o debió tratarse? | Figura 2 · |
| 3. Ponderar la calidad | Construir una puntuación de calidad ponderada por volumen de votos (*Weighted Rating*) y escalar calidad y popularidad para combinarlas con la similitud |¿Cuánto cambia la nota según el volumen de votos? | Figura 3 ·  |
| 4. Implementar, verificar y medir | Implementar el recomendador como un pipeline modular en `src/`, verificado con pruebas, y medir su eficiencia| ¿Qué costo tiene seleccionar los diez mejores? | Figura 4 ·  |
| 5. Evaluar y comunicar| Evaluar las recomendaciones con el criterio medible de la sección 1.4 frente a una línea base aleatoria y comunicar los resultados | ¿El híbrido supera el azar? ¿Mantiene la coherencia de idioma? | Figura 5 ·  |


In [ ]:
# TODO: un registro por objetivo específico de su Fase 1
vinculacion = pd.DataFrame([
    # {"objetivo": "...", "pregunta": "...", "figura": "fig1_...", "grafico": "barras"},
])
vinculacion

### Figura 1

**Pista.** Para una tasa por grupo usen `com.tasa_por_grupo`, que entrega el tamaño y el
intervalo de confianza de cada grupo. Ordenen los grupos por su valor, salvo que tengan un
orden natural (tramos de edad, años).

In [ ]:
figuras = {}

# TODO: reemplazar "su_grupo" y el título por los de su proyecto
# tasas = com.tasa_por_grupo(legible, "su_grupo", OBJETIVO)
# fig, ax = plt.subplots(figsize=(8, 4.4))
# ax.bar(tasas["su_grupo"].astype(str), tasas["tasa"], color=AZUL,
#        yerr=[tasas["tasa"] - tasas["ic_inferior"], tasas["ic_superior"] - tasas["tasa"]],
#        error_kw={"capsize": 4, "lw": 1})
# ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
# ax.set_title("El título dice el hallazgo, no el tipo de gráfico")
# fig.text(0.01, -0.07, FUENTE, fontsize=8, color=GRIS)
# figuras["fig1_..."] = fig
# plt.show()

### Figura 2

**Pista.** Si comparan una distribución entre grupos de tamaño muy distinto, usen
histogramas con `density=True` o diagramas de caja.

In [ ]:
# TODO

### Figura 3

**Pista.** Lista de control: título con hallazgo, ejes con unidades, barras desde cero,
color que señala, tamaño de grupo visible y fuente al pie.

In [ ]:
# TODO

## 5. Asociación no es causalidad

Tomen la asociación más fuerte y compárenla dentro de los estratos de la variable que
podría explicarla. Redacten el resultado con «se asocia con», nunca con «causa».

In [ ]:
# TODO: comparación bruta y estratificada
# bruta = legible.groupby("su_variable")[OBJETIVO].mean()
# estratificada = legible.groupby(["su_estrato", "su_variable"])[OBJETIVO].mean().unstack()

## 6. Validación técnica

Registren controles en los tres escenarios que pide la rúbrica: casos normales, casos
límite y excepciones.

In [ ]:
controles = []


def verificar(escenario, control, condicion, detalle=""):
    controles.append({"escenario": escenario, "control": control,
                      "estado": "OK" if condicion else "FALLA", "detalle": detalle})


verificar("normal", "La copia conserva las filas",
          len(legible) == len(entrenamiento) + len(prueba))

try:
    com.cargar_artefactos(RAIZ / "carpeta_inexistente")
    verificar("excepción", "Falta un artefacto", False)
except FileNotFoundError:
    verificar("excepción", "Falta un artefacto", True, "FileNotFoundError")

# TODO: agreguen al menos un caso normal más y dos casos límite propios de su conjunto

tabla_controles = pd.DataFrame(controles)
assert (tabla_controles["estado"] == "OK").all(), "Hay controles que fallaron"
tabla_controles

## 7. Persistencia

Guarden las figuras a 200 ppp y la tabla de hallazgos en `docs/`. Esa tabla es la fuente
del texto del informe y del guion del video.

In [ ]:
rutas = {nombre: com.guardar_figura(figura, DIR_FIGURAS, nombre) for nombre, figura in figuras.items()}

# TODO: un hallazgo por figura, con la cifra que lo respalda
hallazgos = pd.DataFrame([
    # {"figura": "fig1_...", "hallazgo": "...", "evidencia": "..."},
])
if len(hallazgos):
    hallazgos.to_csv(DIR_DOCS / "hallazgos_f4.csv", index=False)

print("Figuras guardadas:", len(rutas))
print("Hallazgos        :", len(hallazgos))

## Antes de entregar

- [ ] El cuaderno corre completo con **Restart Kernel and Run All**, con numeración continua.
- [ ] Cada objetivo específico tiene al menos una figura, y cada figura responde un objetivo.
- [ ] Todos los títulos dicen el hallazgo.
- [ ] Las asociaciones fuertes se controlaron por al menos una variable de confusión.
- [ ] Las figuras están en `F4/figuras/` y los hallazgos en `docs/hallazgos_f4.csv`.
- [ ] El informe y el video usan las mismas cifras que este cuaderno.
- [ ] Cada integrante tiene commits propios en la Fase 4.